# Step 4b — Does the embedding preserve the meaning of the job-ad sentences?

**Input:** the Step 4 sentence units and embeddings (`outputs/equal_step4_sentence_embeddings/`).
**Output:** validation tables and a stratified human-review file in `outputs/equal_step4b_semantic_validation/`.

There are no gold labels, so semantic preservation is assessed with four label-free or rule-derived checks:

1. **Invariance** — meaning-preserving edits (case, punctuation, bullets, whitespace) should leave a sentence close to itself.
2. **Sensitivity** — meaning-changing edits (gender swap, number change, negation) should move a sentence *away* from itself. A model that scores these as near-identical cannot separate the very distinctions a gender-bias study depends on.
3. **Information probes** — can attributes derived by transparent rules (gender requirement, years of experience, education level) be recovered from the embeddings, compared with TF-IDF? Rule-derived labels are lexical, so TF-IDF is expected to be strong; the probes show what the embedding retains, not human-validated truth.
4. **Human review sample** — neighbour pairs across similarity bands, written to a CSV for manual judgement.

These checks support or limit the use of the embeddings; they do not prove that meaning is preserved.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)

STEP4_DIR = Path("outputs") / "equal_step4_sentence_embeddings"
OUTPUT_DIR = Path("outputs") / "equal_step4b_semantic_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

sentences = pd.read_csv(STEP4_DIR / "step4_sentence_units.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
embeddings = np.load(STEP4_DIR / "step4_sentence_embeddings.npy")
assert len(sentences) == len(embeddings)
model = SentenceTransformer(MODEL_NAME)


def encode(texts):
    return model.encode(list(texts), batch_size=64, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)


# Reference distribution: cosine between random pairs of distinct sentences.
unique = sentences.drop_duplicates("sentence").reset_index(drop=True)
unique_emb = encode(unique["sentence"])
a, b = rng.integers(0, len(unique), 20000), rng.integers(0, len(unique), 20000)
keep = a != b
random_cos = np.einsum("ij,ij->i", unique_emb[a[keep]], unique_emb[b[keep]])
print(f"{len(unique):,} unique sentences. Random-pair cosine: median {np.median(random_cos):.3f}, 95th pct {np.quantile(random_cos, .95):.3f}, 99th pct {np.quantile(random_cos, .99):.3f}")

## 1. Invariance to formatting (should stay close)

For 600 sampled sentences: upper-casing, lower-casing, removing trailing punctuation, adding a bullet marker, and extra whitespace. *Self-retrieval top-1* is the share of perturbed sentences whose nearest neighbour among all unique sentences is their own original.

In [ ]:
sample_idx = rng.choice(len(unique), 600, replace=False)
orig = unique["sentence"].to_numpy()[sample_idx]
perturbations = {
    "lowercase": lambda s: s.lower(),
    "UPPERCASE": lambda s: s.upper(),
    "strip trailing punctuation": lambda s: s.rstrip(".,;:!? "),
    "add bullet marker": lambda s: "• " + s,
    "extra whitespace": lambda s: s.replace(" ", "   "),
}
rows = []
p99 = np.quantile(random_cos, .99)
for name, fn in perturbations.items():
    vec = encode([fn(s) for s in orig])
    cos = np.einsum("ij,ij->i", vec, unique_emb[sample_idx])
    top1 = (np.argmax(vec @ unique_emb.T, axis=1) == sample_idx).mean()
    rows.append({"perturbation": name, "n": len(cos), "median_cosine": np.median(cos), "p5_cosine": np.quantile(cos, .05),
                 "share_above_random_p99": (cos > p99).mean(), "self_retrieval_top1": top1})
invariance = pd.DataFrame(rows).round(3)
display(invariance)

## 2. Sensitivity to meaning changes (should move away)

Pairs are built from real corpus sentences by a single targeted edit:

- **gender swap** — `male` ↔ `female` (sentences with exactly one of the two words);
- **number change** — the first number before *year(s)* is increased by 4 (e.g. “1 year” → “5 year”);
- **negation** — a leading `With ` becomes `Without `;
- **control: formatting only** — lowercase, from section 1.

If a meaning-changing edit scores as high as a formatting-only edit, the embedding does not separate that distinction.

In [ ]:
def swap_gender(s):
    has_m, has_f = re.search(r"\bmale\b", s, re.I), re.search(r"\bfemale\b", s, re.I)
    if bool(has_m) == bool(has_f):
        return None
    def repl(m):
        w = m.group(0)
        new = "female" if w.lower() == "male" else "male"
        return new.upper() if w.isupper() else new.capitalize() if w[0].isupper() else new
    return re.sub(r"\b(?:fe)?male\b", repl, s, flags=re.I)


def change_number(s):
    m = re.search(r"\b(\d{1,2})(?=\s*(?:\+\s*)?years?\b)", s, re.I)
    return None if not m else s[: m.start()] + str(int(m.group(1)) + 4) + s[m.end():]


def negate_with(s):
    return re.sub(r"^With\b(?! or without)", "Without", s) if re.match(r"^With\b(?! or without)", s) else None


edits = {"gender swap (male <-> female)": swap_gender, "number change (years + 4)": change_number, "negation (With -> Without)": negate_with}
pair_rows, example_rows = [], []
for name, fn in edits.items():
    pairs = [(s, fn(s)) for s in unique["sentence"] if fn(s) and fn(s) != s]
    pairs = [pairs[i] for i in rng.permutation(len(pairs))[:300]]
    v1, v2 = encode([p[0] for p in pairs]), encode([p[1] for p in pairs])
    cos = np.einsum("ij,ij->i", v1, v2)
    pair_rows.append({"edit": name, "n_pairs": len(pairs), "median_cosine": np.median(cos), "p5_cosine": np.quantile(cos, .05), "p95_cosine": np.quantile(cos, .95),
                      "share_above_lowercase_median": (cos >= invariance.loc[invariance["perturbation"] == "lowercase", "median_cosine"].iloc[0]).mean()})
    for (s1, s2), c in list(zip(pairs, cos))[:3]:
        example_rows.append({"edit": name, "original": s1, "edited": s2, "cosine": round(float(c), 3)})
sensitivity = pd.DataFrame(pair_rows).round(3)
sensitivity.loc[len(sensitivity)] = ["control: formatting only (lowercase)", 600, *invariance.loc[invariance["perturbation"] == "lowercase", ["median_cosine", "p5_cosine"]].iloc[0], np.nan, np.nan]
display(sensitivity)
display(pd.DataFrame(example_rows))
print(f"Reference: random-pair cosine median {np.median(random_cos):.3f}, 95th percentile {np.quantile(random_cos, .95):.3f}")

## 3. Information probes

Unique sentences are labelled by transparent rules, then a logistic-regression probe is trained on (a) the embeddings and (b) TF-IDF features, with 5-fold cross-validation **grouped by employer** so one employer's templates do not appear in both train and test. Macro-F1 is reported next to a majority-class baseline. Because the labels are lexical, TF-IDF is expected to do well; a large gap in the other direction would indicate information the embedding loses.

In [ ]:
def gender_label(s):
    m, f = bool(re.search(r"\bmale\b|\bmen\b|\bman\b|\bboys?\b", s, re.I)), bool(re.search(r"\bfemale\b|\bwomen\b|\bwoman\b|\bgirls?\b|\blady\b|\bladies\b", s, re.I))
    return "both" if m and f else "male_only" if m else "female_only" if f else None


def experience_label(s):
    m = re.search(r"\b(\d{1,2})\s*(?:to|-|–)?\s*(?:\d{1,2})?\s*(?:\+\s*)?years?\b", s, re.I)
    if not m:
        return None
    n = int(m.group(1))
    return "0-1 years" if n <= 1 else "2-3 years" if n <= 3 else "4+ years"


def education_label(s):
    if re.search(r"\bhigh school\b|\bsenior high\b|\bshs\b", s, re.I):
        return "high_school"
    if re.search(r"\bvocational\b|\btesda\b|\btechnical\b", s, re.I):
        return "vocational"
    if re.search(r"\bcollege\b|\bbachelor\b|\bdegree\b|\bgraduate of\b|\bBS\b|\bBSBA\b", s):
        return "college_degree"
    return None


tasks = {"gender requirement wording": gender_label, "years of experience": experience_label, "education level": education_label}
tfidf_all = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
rows = []
for task, fn in tasks.items():
    labels = unique["sentence"].map(fn)
    mask = labels.notna().to_numpy()
    y, groups, texts, X_emb = labels[mask].to_numpy(), unique.loc[mask, "employer_id"].to_numpy(), unique.loc[mask, "sentence"], unique_emb[mask]
    counts = pd.Series(y).value_counts()
    scores = {"embedding": [], "tfidf": [], "majority": []}
    for train, test in GroupKFold(n_splits=5).split(X_emb, y, groups):
        if len(set(y[train])) < 2:
            continue
        clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_emb[train], y[train])
        scores["embedding"].append(f1_score(y[test], clf.predict(X_emb[test]), average="macro"))
        vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(texts.iloc[train])
        clf2 = LogisticRegression(max_iter=2000, class_weight="balanced").fit(vec.transform(texts.iloc[train]), y[train])
        scores["tfidf"].append(f1_score(y[test], clf2.predict(vec.transform(texts.iloc[test])), average="macro"))
        scores["majority"].append(f1_score(y[test], np.full(len(test), pd.Series(y[train]).mode()[0]), average="macro"))
    rows.append({"task": task, "n_sentences": len(y), "class_counts": dict(counts), "embedding_macro_f1": np.mean(scores["embedding"]),
                 "tfidf_macro_f1": np.mean(scores["tfidf"]), "majority_macro_f1": np.mean(scores["majority"])})
probes = pd.DataFrame(rows)
probes[["embedding_macro_f1", "tfidf_macro_f1", "majority_macro_f1"]] = probes[["embedding_macro_f1", "tfidf_macro_f1", "majority_macro_f1"]].round(3)
display(probes)

## 4. Human-review sample

For 120 random unique sentences, the nearest *other* unique sentence is paired with it. Pairs are stratified into similarity bands (cosine ≥ 0.9, 0.8–0.9, 0.7–0.8, < 0.7) so a reviewer can judge whether the model's similarity matches meaning at each level. `reviewer_judgement` is left empty (suggested values: `same meaning`, `related`, `different`).

In [ ]:
query_idx = rng.choice(len(unique), 600, replace=False)
sim = unique_emb[query_idx] @ unique_emb.T
sim[np.arange(len(query_idx)), query_idx] = -1
nn = sim.argmax(axis=1)
pairs = pd.DataFrame({"sentence_a": unique["sentence"].to_numpy()[query_idx], "sentence_b": unique["sentence"].to_numpy()[nn], "cosine": sim[np.arange(len(query_idx)), nn]})
pairs["band"] = pd.cut(pairs["cosine"], [-1, .7, .8, .9, 1.01], labels=["<0.7", "0.7-0.8", "0.8-0.9", ">=0.9"], right=False)
print(pairs["band"].value_counts().sort_index().to_string())
review = pairs.groupby("band", observed=True, group_keys=False).apply(lambda g: g.sample(min(30, len(g)), random_state=RANDOM_STATE)).reset_index(drop=True)
review["cosine"] = review["cosine"].round(3)
review["reviewer_judgement"] = ""
display(review.groupby("band", observed=True).head(3))

invariance.to_csv(OUTPUT_DIR / "step4b_invariance.csv", index=False, encoding="utf-8-sig")
sensitivity.to_csv(OUTPUT_DIR / "step4b_sensitivity.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(example_rows).to_csv(OUTPUT_DIR / "step4b_sensitivity_examples.csv", index=False, encoding="utf-8-sig")
probes.to_csv(OUTPUT_DIR / "step4b_probes.csv", index=False, encoding="utf-8-sig")
review.to_csv(OUTPUT_DIR / "step4b_human_review_pairs.csv", index=False, encoding="utf-8-sig")
print("Saved:", *sorted(p.name for p in OUTPUT_DIR.iterdir()), sep="\n  ")